# Lab 2.3 — Quantization: Post-Training vs Quantization-Aware Training

This notebook has `# TODO` markers (TODO 1-6). Work through them in order.

Copy your `mnist_cnn.h5` from Lab 2.2 into this same folder before you start.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras
import matplotlib.pyplot as plt

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

## Step 0 — Load data and the Lab 2.2 model

Same MNIST preprocessing as Lab 2.2 (normalize to [0,1], add channel dim).
This part is provided for you.

Note: loaded with `tf_keras.models.load_model` (not `tf.keras.models.load_model`)
— matching the `tf_keras` your Lab 2.2 model was built and saved with.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train_n = (x_train / 255.0).astype("float32")[..., np.newaxis]
x_test_n  = (x_test  / 255.0).astype("float32")[..., np.newaxis]

model = tf_keras.models.load_model("mnist_cnn.h5")
model.summary()

## Step 1 — Baseline

**TODO 1:** Record the float32 model's size on disk (`os.path.getsize`, in KB)
and its test accuracy (`model.evaluate`). Also measure latency: the average
time for a single-image forward pass, calling the model directly
(`model(x, training=False)`) rather than `.predict()`, over 100 images.

In [ ]:
# TODO 1: size (KB), accuracy, and per-image latency (ms) of the baseline model
baseline_size_kb = None
baseline_acc = None

n_latency = 100
start = time.time()
for i in range(n_latency):
    pass  # TODO: call the model on a single image, x_test_n[i:i+1]
baseline_latency_ms = None

print(f"Baseline float32 -- size: {baseline_size_kb:.1f} KB, "
      f"accuracy: {baseline_acc:.4f}, latency: {baseline_latency_ms:.3f} ms/image")

## Step 2 — Post-Training Quantization (PTQ)

`tf.lite.TFLiteConverter` needs a **representative dataset** — a small sample
of real inputs — to figure out the right scale/zero-point for quantizing
activations. A few hundred training images is plenty (provided for you
below).

**TODO 2:** Configure the converter for full-integer quantization:
- `optimizations = [tf.lite.Optimize.DEFAULT]`
- `representative_dataset = representative_dataset`
- `target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]`
- `inference_input_type = tf.int8`
- `inference_output_type = tf.int8`

The last two matter for Lab 2.4 — the microcontroller runtime expects int8
tensors in and out, not just int8 weights internally.

In [ ]:
def representative_dataset():
    for i in range(200):
        yield [x_train_n[i:i+1]]

# TODO 2: build and configure the converter, then convert and save ptq_model.tflite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_ptq = None

with open("ptq_model.tflite", "wb") as f:
    f.write(tflite_ptq)

ptq_size_kb = len(tflite_ptq) / 1024
print(f"PTQ model size: {ptq_size_kb:.1f} KB "
      f"({baseline_size_kb / ptq_size_kb:.1f}x smaller than float32)")

**Evaluating a TFLite model is different from evaluating a Keras model** —
you drive the `Interpreter` directly, and since the input tensor is now
`int8`, you have to quantize each image yourself using the input tensor's
`(scale, zero_point)` before feeding it in:

```
quantized_pixel = round(pixel_value / scale + zero_point)
```

The output is int8 too, but since quantization is a monotonic
(order-preserving) transform, `argmax` on the raw int8 output gives the same
predicted class as `argmax` on the dequantized version — no need to
dequantize just to get a prediction.

**TODO 3:** Complete `evaluate_tflite()`: quantize each input, run it through
the interpreter, and check whether the predicted class matches the true
label. Also time 100 invocations for a latency figure, the same way as
Step 1.

In [ ]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(img):
        # TODO 3a: quantize a float image using (scale, zero_point)
        return None

    # TODO 3b: loop over all of x/y, run inference, count correct predictions
    correct = 0
    accuracy = None

    # TODO 3c: time n_latency invocations (same pattern as Step 1)
    latency_ms = None

    return accuracy, latency_ms, inp["dtype"], out["dtype"]

ptq_acc, ptq_latency_ms, ptq_in_dtype, ptq_out_dtype = evaluate_tflite(tflite_ptq, x_test_n, y_test)
print(f"PTQ -- accuracy: {ptq_acc:.4f}, latency: {ptq_latency_ms:.3f} ms/image, "
      f"input dtype: {ptq_in_dtype}, output dtype: {ptq_out_dtype}")

## Step 3 — Quantization-Aware Training (QAT)

`tensorflow_model_optimization` depends on the `tf_keras` package (legacy
Keras 2) rather than the Keras 3 that's `tf.keras` by default in recent
TensorFlow — `quantize_model()` specifically requires a `tf_keras`
`Sequential`/`Functional` model. Since `model` was already loaded via
`tf_keras.models.load_model` in Step 0 (matching how it was built and saved
in Lab 2.2), you can wrap it directly — no rebuilding, no copying weights.

**TODO 4:** Import `tensorflow_model_optimization as tfmot`, wrap `model`
with `tfmot.quantization.keras.quantize_model()`, compile it (same
loss/metric as Step 1), and fine-tune for 2 epochs.

In [ ]:
# TODO 4: import tfmot, wrap `model`, compile, and fine-tune
qat_model = None

**TODO 5:** Convert `qat_model` to a fully-quantized int8 TFLite model —
same converter settings as Step 2 — and evaluate it with your
`evaluate_tflite()` from Step 2.

In [ ]:
# TODO 5: convert qat_model to int8 TFLite, save as qat_model.tflite, then evaluate
tflite_qat = None

with open("qat_model.tflite", "wb") as f:
    f.write(tflite_qat)

qat_size_kb = len(tflite_qat) / 1024
qat_acc, qat_latency_ms, qat_in_dtype, qat_out_dtype = evaluate_tflite(tflite_qat, x_test_n, y_test)
print(f"QAT -- size: {qat_size_kb:.1f} KB, accuracy: {qat_acc:.4f}, "
      f"latency: {qat_latency_ms:.3f} ms/image, "
      f"input dtype: {qat_in_dtype}, output dtype: {qat_out_dtype}")

## Step 4 — Compare

Check that both quantized models have `int8` input/output tensors printed
above — that's the requirement Lab 2.4's microcontroller runtime needs.

**TODO 6:** Build a comparison table (a small `pandas.DataFrame` is easiest)
with one row per variant (float32 baseline, PTQ int8, QAT int8) and columns
for size (KB), accuracy, and latency (ms). Then plot size and accuracy side
by side as bar charts.

In [ ]:
# TODO 6: build the comparison DataFrame
comparison = None
comparison

In [ ]:
# TODO 6 (continued): bar charts for size and accuracy, side by side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

plt.tight_layout()
plt.savefig("quantization_comparison.png", dpi=110)
plt.show()

## Discussion

In one paragraph: did QAT actually beat PTQ here? By how much, and was the
extra training time worth it for your use case?